In [2]:
from pathlib import Path
import pandas as pd
import optuna
from optuna.storages import JournalStorage, JournalFileStorage

LOG_FILE = Path("./hp_search.log")

storage = JournalStorage(JournalFileStorage(str(LOG_FILE)))
study = optuna.load_study(study_name="prob_mlp_hp_search", storage=storage)

print(f"Trials completed : {len(study.trials)}")
print(f"Best val MSE     : {study.best_value:.6f}")
print(f"Best params      : {study.best_params}")

Trials completed : 33
Best val MSE     : 0.000041
Best params      : {'n_layers': 3, 'hidden_dim': 32, 'lr': 0.009678555772799668, 'batch_size': 512, 'dropout': 0.0}


/tmp/ipykernel_327704/574621968.py:8: FutureWarning: JournalFileStorage has been deprecated in v4.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.0.0. Use :class:`~optuna.storages.journal.JournalFileBackend` instead.
  storage = JournalStorage(JournalFileStorage(str(LOG_FILE)))


In [3]:
df = study.trials_dataframe()

# Keep only useful columns and rename for clarity
param_cols = [c for c in df.columns if c.startswith("params_")]
keep = ["number", "value", "state", "duration"] + param_cols
df = df[keep].rename(columns={
    "number": "trial",
    "value":  "val_mse",
    **{c: c.replace("params_", "") for c in param_cols}
})

# Sort by val_mse ascending (best first), show only completed trials
df_done = df[df["state"] == "COMPLETE"].sort_values("val_mse").reset_index(drop=True)
df_done

,trial,val_mse,state,duration,batch_size,dropout,hidden_dim,lr,n_layers
0,21,0.000041,COMPLETE,0 days 00:00:08.675577,512,0.0,32,0.009679,3
1,23,0.000041,COMPLETE,0 days 00:00:12.830510,512,0.0,32,0.003231,3
2,2,0.000052,COMPLETE,0 days 00:00:14.576321,512,0.0,16,0.007025,3
3,31,0.000053,COMPLETE,0 days 00:00:07.655070,512,0.0,32,0.009717,3
4,20,0.000053,COMPLETE,0 days 00:00:07.366375,512,0.0,32,0.009944,3
5,16,0.000058,COMPLETE,0 days 00:00:08.582432,512,0.0,32,0.004746,2
6,5,0.000063,COMPLETE,0 days 00:00:11.625128,512,0.0,32,0.001538,4
7,22,0.000064,COMPLETE,0 days 00:00:04.190600,512,0.0,32,0.008471,3
8,14,0.000065,COMPLETE,0 days 00:00:10.265311,512,0.0,16,0.002065,4
9,12,0.000068,COMPLETE,0 days 00:00:09.933553,512,0.0,16,0.001801,4


In [ ]:
# Top 10 trials
df_done.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Val MSE over trials
axes[0].plot(df_done["trial"], df_done["val_mse"], "o", alpha=0.5, markersize=4)
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Val MSE")
axes[0].set_title("Val MSE per trial")
axes[0].set_yscale("log")

# Best val MSE over time (running minimum)
ordered = df.loc[df["state"] == "COMPLETE"].sort_values("trial")
axes[1].plot(ordered["trial"], ordered["val_mse"].cummin(), "-")
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Best val MSE so far")
axes[1].set_title("Optimisation progress")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# Parameter importance
importances = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame(importances.items(), columns=["param", "importance"]).sort_values("importance", ascending=False)

imp_df.plot(kind="barh", x="param", y="importance", legend=False,
            title="HP importance (FAnova)", figsize=(7, 4))
plt.tight_layout()
plt.show()
imp_df